# Perturbation Panel Optimization

## Goal
Find the **minimal set of background perturbations** that maximizes causal identifiability
across the 55 unidentifiable baseline queries in the *E. coli* 50-gene experimental set.

## Why are queries unidentifiable? The cycle problem

The *E. coli* transcriptional regulatory network (TRN) is not a DAG — it contains
**feedback loops** (cycles). The cyclic ID algorithm can prove whether a causal effect
$P(Y \mid \mathrm{do}(X))$ is identifiable from observational data, but when $X$ or $Y$
participates in a cycle, the algorithm often cannot uniquely solve for the effect:
the cycle creates an algebraic dependency that makes the system under-determined.

**Key structural insight:** A "hard" intervention $\mathrm{do}(g)$ *removes all incoming
edges to $g$* in the causal graph. This **breaks every cycle that passes through $g$**.
Therefore, TFs that participate in many cycles are the highest-priority background
perturbation candidates — intervening on them structurally simplifies the graph and
unlocks the most previously-unidentifiable queries.

We quantify this with the **cycle-breaking score**: the number of simple cycles in the
TRN that contain a given node. For the full E. coli graph (>500 nodes), we use the
strongly-connected component (SCC) size as a tractable proxy (SCC size − 1, so
singleton nodes score 0).

## Mathematical framing
This is a **submodular maximum-coverage / minimum-set-cover** problem.

Define, for each candidate TF `g` (any regulator with out-degree ≥ 1, ~250 genes):

$$\text{Cover}(g) = \{ q \in \text{Unidentifiable} : \text{cyclic\_id resolves } q \text{ when } \mathrm{do}(g) \text{ is added as background} \}$$

Two optimization objectives:
- **Budgeted max-coverage** (given budget $k$): choose $S \subseteq$ candidates, $|S| \le k$, to maximize $|\bigcup_{g \in S} \text{Cover}(g)|$.
  Greedy achieves a provable $(1 - 1/e) \approx 63\%$ of optimal.
- **Min-set-cover**: fewest TFs to cover all resolvable queries.
  Greedy achieves $\ln(n)$ approximation.

**Note on replicates:** Identifiability is a *structural* property of the causal graph —
it is independent of sample size. Replicate count (100s–1000s) governs estimation
precision once a query is identifiable, not whether it is identifiable.

## Workflow
1. **Step 1 (offline):** Run `scripts/build_coverage_matrix.py` to generate `coverage_matrix.csv`.
   Candidates are pre-ranked by cycle-breaking score so the most valuable TFs are
   evaluated first. This calls `cyclic_id` ~13,000–14,000 times (55 queries × ~250 TF
   candidates) and checkpoints every 100 evaluations.
2. **Step 2 (this notebook):** Load the matrix, analyse cycle structure, run the greedy
   optimizer, visualize results.

---

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add scripts/ to path so we can import the optimizer functions directly
repo_root = Path("../..").resolve()
sys.path.insert(0, str(repo_root / "scripts"))

from perturbation_optimizer import (
    build_marginal_gain_curve,
    greedy_max_coverage,
    greedy_min_set_cover,
    load_coverage_matrix,
)

# Paths
NOTEBOOK_DIR = Path(".").resolve()
MATRIX_PATH = NOTEBOOK_DIR / "coverage_matrix.csv"
OUTPUT_DIR = NOTEBOOK_DIR

BUDGETS = [2, 5, 10, 15, 20, 25]
MAX_K = max(BUDGETS)

# Wong color-blind palette
BLUE = "#0072B2"
ORANGE = "#D55E00"
GREEN = "#009E73"
YELLOW = "#F0E442"
PURPLE = "#CC79A7"

print(f"Matrix path: {MATRIX_PATH}")
print(f"Matrix exists: {MATRIX_PATH.exists()}")

## 1. Load Coverage Matrix

If `coverage_matrix.csv` does not yet exist, run the build script first:

```bash
uv run python scripts/build_coverage_matrix.py \
  --graphml notebooks/Ecoli_Analysis_Notebooks/ecoli_full_network_no_small_rna.graphml \
  --supptable notebooks/Ecoli_Analysis_Notebooks/supptable1.csv \
  --output notebooks/Ecoli_Analysis_Notebooks/coverage_matrix.csv \
  --checkpoint notebooks/Ecoli_Analysis_Notebooks/coverage_checkpoint.json
```

This is a long-running job (~13k `cyclic_id` calls). It checkpoints every 100 evaluations
and is fully resumable.

In [ ]:
if not MATRIX_PATH.exists():
    raise FileNotFoundError(
        f"coverage_matrix.csv not found at {MATRIX_PATH}.\n"
        "Run scripts/build_coverage_matrix.py first (see instructions above)."
    )

candidates, queries, matrix = load_coverage_matrix(str(MATRIX_PATH))
n_queries = len(queries)
n_candidates = len(candidates)

# Count resolvable queries
resolvable_idx = set()
for cand in candidates:
    for qi, val in enumerate(matrix[cand]):
        if val:
            resolvable_idx.add(qi)
n_resolvable = len(resolvable_idx)

print(f"Candidate TFs:              {n_candidates}")
print(f"Unidentifiable queries:     {n_queries}")
print(f"Resolvable by >= 1 TF:      {n_resolvable} ({n_resolvable / n_queries * 100:.1f}%)")
print(f"Permanently unresolvable:   {n_queries - n_resolvable}")

## 2. Coverage Distribution

How many queries does each candidate TF resolve on its own?

In [ ]:
# Per-TF coverage counts
coverage_counts = {cand: sum(matrix[cand]) for cand in candidates}
counts_series = pd.Series(coverage_counts).sort_values(ascending=False)

print("Top 20 candidate TFs by single-background coverage:")
print(counts_series.head(20).to_string())
print(f"\nTFs with coverage > 0: {(counts_series > 0).sum()} / {n_candidates}")
print(f"TFs with coverage = 0: {(counts_series == 0).sum()} / {n_candidates}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of per-TF coverage
ax = axes[0]
nonzero = counts_series[counts_series > 0]
ax.hist(
    nonzero.values,
    bins=range(0, int(nonzero.max()) + 2),
    color=BLUE,
    edgecolor="white",
    align="left",
)
ax.set_xlabel("Queries resolved by single background TF", fontsize=11)
ax.set_ylabel("Number of candidate TFs", fontsize=11)
ax.set_title("Distribution of Single-TF Coverage", fontsize=12, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Right: top-20 bar chart
ax2 = axes[1]
top20 = counts_series.head(20)
bars = ax2.barh(top20.index[::-1], top20.values[::-1], color=BLUE, edgecolor="white")
for bar, val in zip(bars, top20.values[::-1], strict=False):
    ax2.text(
        bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2, str(val), va="center", fontsize=9
    )
ax2.set_xlabel("Queries resolved", fontsize=11)
ax2.set_title("Top 20 Candidate TFs by Coverage", fontsize=12, fontweight="bold")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "coverage_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: coverage_distribution.png")

## 2.5 Cycle-Breaking Analysis: The Structural Basis for Ranking

### Why cycle membership predicts coverage

A query $P(Y \mid \mathrm{do}(X))$ is unidentifiable when $X$ or $Y$ sits inside a
feedback loop that the ID algorithm cannot factor out. A hard intervention
$\mathrm{do}(g)$ severs **all incoming edges** to $g$, removing $g$ from every cycle
it participates in. This is why cycle membership is a strong *a priori* predictor of
how many queries a background perturbation will rescue.

We compute two cycle-breaking metrics:
- **Exact cycle count** (small graphs): `nx.simple_cycles` — number of simple cycles
  containing the node.
- **SCC-size proxy** (large graphs, used here): size of the node's strongly-connected
  component minus 1. Nodes in larger SCCs participate in more cycles; singleton nodes
  (SCC size = 1) are not in any cycle and score 0.

We then compare the cycle-score ranking to the empirical coverage ranking to validate
the structural hypothesis.

In [ ]:
import sys
from pathlib import Path

import networkx as nx

# Import cycle-breaking functions

# Load the E. coli graph (needed for cycle analysis)
GRAPHML_PATH = NOTEBOOK_DIR / "ecoli_full_network_no_small_rna.graphml"
if not GRAPHML_PATH.exists():
    print(f"WARNING: {GRAPHML_PATH} not found — skipping cycle analysis.")
    ecoli_graph = None
else:
    ecoli_graph = nx.read_graphml(str(GRAPHML_PATH))
    print(
        f"Graph loaded: {ecoli_graph.number_of_nodes()} nodes, "
        f"{ecoli_graph.number_of_edges()} edges"
    )

    # Compute SCC sizes for all candidate TFs
    scc_map = {}
    for scc in nx.strongly_connected_components(ecoli_graph):
        scc_size = len(scc)
        for node in scc:
            scc_map[node] = scc_size - 1  # 0 for singletons

    cycle_scores = {cand: scc_map.get(cand, 0) for cand in candidates}
    cycle_series = pd.Series(cycle_scores).sort_values(ascending=False)

    print(
        f"\nCandidates in a non-trivial SCC (cycle score > 0): "
        f"{(cycle_series > 0).sum()} / {len(candidates)}"
    )
    print(
        f"Candidates in singleton SCC (cycle score = 0):    "
        f"{(cycle_series == 0).sum()} / {len(candidates)}"
    )
    print("\nTop 10 by cycle-breaking score (SCC proxy):")
    print(cycle_series.head(10).to_string())

In [ ]:
if ecoli_graph is not None:
    # Build a combined DataFrame: cycle score + coverage count
    combined_df = (
        pd.DataFrame(
            {
                "candidate_tf": candidates,
                "coverage_count": [coverage_counts[c] for c in candidates],
                "cycle_score": [cycle_scores[c] for c in candidates],
            }
        )
        .sort_values("cycle_score", ascending=False)
        .reset_index(drop=True)
    )

    combined_df["coverage_rank"] = (
        combined_df["coverage_count"].rank(ascending=False, method="min").astype(int)
    )
    combined_df["cycle_rank"] = (
        combined_df["cycle_score"].rank(ascending=False, method="min").astype(int)
    )

    combined_df.to_csv(OUTPUT_DIR / "cycle_breaking_analysis.csv", index=False)

    print("Top 20 candidates by cycle-breaking score:")
    display(
        combined_df[
            ["candidate_tf", "cycle_score", "cycle_rank", "coverage_count", "coverage_rank"]
        ].head(20)
    )

In [ ]:
if ecoli_graph is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # --- Left: Top-20 by cycle score vs. coverage count (grouped bar) ---
    ax = axes[0]
    top20_cycle = combined_df.head(20).copy()
    x = np.arange(len(top20_cycle))
    width = 0.4

    # Normalize both to [0,1] for visual comparison
    max_cycle = top20_cycle["cycle_score"].max() or 1
    max_cov = top20_cycle["coverage_count"].max() or 1

    bars1 = ax.bar(
        x - width / 2,
        top20_cycle["cycle_score"] / max_cycle,
        width,
        color=PURPLE,
        label="Cycle-breaking score (norm.)",
        edgecolor="white",
        alpha=0.85,
    )
    bars2 = ax.bar(
        x + width / 2,
        top20_cycle["coverage_count"] / max_cov,
        width,
        color=BLUE,
        label="Coverage count (norm.)",
        edgecolor="white",
        alpha=0.85,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(top20_cycle["candidate_tf"], rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Normalized score", fontsize=11)
    ax.set_title(
        "Top 20 by Cycle Score:\nCycle-Breaking vs. Empirical Coverage",
        fontsize=12,
        fontweight="bold",
    )
    ax.legend(fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # --- Right: Scatter — cycle score vs. coverage count ---
    ax2 = axes[1]
    sc = ax2.scatter(
        combined_df["cycle_score"],
        combined_df["coverage_count"],
        c=combined_df["coverage_count"],
        cmap="Blues",
        edgecolors="gray",
        linewidths=0.4,
        s=50,
        alpha=0.8,
    )
    plt.colorbar(sc, ax=ax2, label="Coverage count")

    # Label top-5 by coverage
    top5_cov = combined_df.nlargest(5, "coverage_count")
    for _, row in top5_cov.iterrows():
        ax2.annotate(
            row["candidate_tf"],
            xy=(row["cycle_score"], row["coverage_count"]),
            xytext=(4, 2),
            textcoords="offset points",
            fontsize=8,
            color=ORANGE,
        )

    # Pearson correlation
    corr = combined_df[["cycle_score", "coverage_count"]].corr().iloc[0, 1]
    ax2.set_xlabel("Cycle-breaking score (SCC size proxy)", fontsize=11)
    ax2.set_ylabel("Queries resolved (single background)", fontsize=11)
    ax2.set_title(
        f"Cycle Score vs. Empirical Coverage\nPearson r = {corr:.3f}",
        fontsize=12,
        fontweight="bold",
    )
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "cycle_breaking_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: cycle_breaking_analysis.png")
    print(f"Pearson correlation (cycle score vs. coverage): {corr:.3f}")

In [ ]:
if ecoli_graph is not None:
    # Side-by-side ranking comparison: cycle-score order vs. coverage order
    top_n = 15
    by_cycle = combined_df.nlargest(top_n, "cycle_score")[
        ["candidate_tf", "cycle_score", "coverage_count"]
    ].reset_index(drop=True)
    by_coverage = combined_df.nlargest(top_n, "coverage_count")[
        ["candidate_tf", "cycle_score", "coverage_count"]
    ].reset_index(drop=True)

    comparison = pd.DataFrame(
        {
            "Rank": range(1, top_n + 1),
            "By cycle score": by_cycle["candidate_tf"],
            "Cycle score": by_cycle["cycle_score"],
            "By coverage count": by_coverage["candidate_tf"],
            "Coverage count": by_coverage["coverage_count"],
        }
    )

    # Highlight TFs that appear in both top-15 lists
    overlap = set(by_cycle["candidate_tf"]) & set(by_coverage["candidate_tf"])
    print(
        f"Top-{top_n} overlap between cycle-score and coverage rankings: {len(overlap)}/{top_n} TFs"
    )
    print(f"Overlapping TFs: {sorted(overlap)}")
    print()
    print("Side-by-side ranking comparison:")
    display(comparison)

    comparison.to_csv(OUTPUT_DIR / "cycle_vs_coverage_ranking.csv", index=False)
    print("Saved: cycle_vs_coverage_ranking.csv")

### Interpretation

- **High correlation** between cycle score and coverage count validates the structural
  hypothesis: TFs that break more cycles empirically rescue more unidentifiable queries.
- **Discrepancies** (high cycle score but low coverage, or vice versa) reveal cases where
  cycle membership alone is insufficient — the specific topology of the cycle relative to
  the query pair matters. These are candidates for deeper structural analysis.
- **Practical implication for `build_coverage_matrix.py`:** Candidates are pre-sorted by
  cycle-breaking score before the expensive `cyclic_id` evaluation loop. This means
  early checkpoint snapshots already contain the highest-value candidates, and the
  greedy optimizer can be run on a partial matrix if the full job is interrupted.

## 3. Greedy Max-Coverage: Marginal Gain Curve

How many queries are unlocked as we add more perturbations to the panel?

In [ ]:
# Run greedy up to MAX_K
full_greedy = greedy_max_coverage(candidates, queries, matrix, MAX_K)
curve = build_marginal_gain_curve(candidates, queries, matrix, MAX_K)

curve_df = pd.DataFrame(curve, columns=["k", "queries_resolved", "fraction_resolved"])
curve_df["pct_resolved"] = curve_df["fraction_resolved"] * 100

print("Marginal gain curve (greedy max-coverage):")
print(curve_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    curve_df["k"],
    curve_df["queries_resolved"],
    color=BLUE,
    linewidth=2.5,
    marker="o",
    markersize=6,
    label="Greedy (optimal order)",
)

# Mark the budget checkpoints
for k in BUDGETS:
    row = curve_df[curve_df["k"] == k]
    if not row.empty:
        y = row["queries_resolved"].values[0]
        ax.axvline(k, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
        ax.annotate(
            f"k={k}\n{y} queries", xy=(k, y), xytext=(k + 0.3, y - 1.5), fontsize=8, color="gray"
        )

# Ceiling: max resolvable
ax.axhline(
    n_resolvable,
    color=ORANGE,
    linestyle=":",
    linewidth=1.5,
    label=f"Max resolvable ({n_resolvable}/{n_queries})",
)

ax.set_xlabel("Number of background perturbations (k)", fontsize=12)
ax.set_ylabel("Unidentifiable queries resolved", fontsize=12)
ax.set_title(
    "Information Gain vs. Number of Simultaneous Perturbations\n"
    "E. coli Gene Regulatory Network — Greedy Optimal Panel",
    fontsize=13,
    fontweight="bold",
    pad=15,
)
ax.set_xlim(0, MAX_K + 1)
ax.set_ylim(0, n_queries * 1.1)
ax.legend(fontsize=11)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "marginal_gain_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: marginal_gain_curve.png")

## 4. Nominated Panel per Budget k

For each budget k ∈ {2, 5, 10, 15, 20, 25}, show the ordered nomination list.

In [ ]:
nomination_tables = {}

for k in BUDGETS:
    selected_k = full_greedy[:k]
    rows = []
    for rank, (tf, gain, cumulative) in enumerate(selected_k, start=1):
        rows.append(
            {
                "Rank": rank,
                "Candidate TF": tf,
                "Marginal gain": gain,
                "Cumulative resolved": cumulative,
                "% of unidentifiable": f"{cumulative / n_queries * 100:.1f}%",
            }
        )
    df = pd.DataFrame(rows)
    nomination_tables[k] = df
    df.to_csv(OUTPUT_DIR / f"nomination_k{k:02d}.csv", index=False)

# Print summary table
print("=" * 60)
print("NOMINATION SUMMARY — top TF per budget")
print("=" * 60)
print(f"{'k':>4}  {'Top TF':<20}  {'Resolved':>8}  {'% of 55':>8}")
print("-" * 50)
for k in BUDGETS:
    df = nomination_tables[k]
    if df.empty:
        print(f"{k:>4}  {'(none)':20}  {'0':>8}  {'0.0%':>8}")
    else:
        last = df.iloc[-1]
        top = df.iloc[0]
        print(
            f"{k:>4}  {top['Candidate TF']:<20}  "
            f"{last['Cumulative resolved']:>8}  "
            f"{last['% of unidentifiable']:>8}"
        )

In [ ]:
# Show the full nomination table for the recommended budget (k=10)
RECOMMENDED_K = 10
print(f"Full nomination table for k={RECOMMENDED_K}:")
display(nomination_tables[RECOMMENDED_K])

## 5. Budget Comparison Bar Chart

How many queries are resolved at each budget checkpoint?

In [ ]:
budget_resolved = []
for k in BUDGETS:
    df = nomination_tables[k]
    resolved = df["Cumulative resolved"].max() if not df.empty else 0
    budget_resolved.append(resolved)

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar([str(k) for k in BUDGETS], budget_resolved, color=BLUE, edgecolor="white", width=0.6)

# Ceiling line
ax.axhline(
    n_resolvable,
    color=ORANGE,
    linestyle="--",
    linewidth=1.5,
    label=f"Max resolvable ({n_resolvable})",
)
ax.axhline(
    n_queries,
    color="gray",
    linestyle=":",
    linewidth=1.2,
    label=f"Total unidentifiable ({n_queries})",
)

for bar, val in zip(bars, budget_resolved, strict=False):
    pct = val / n_queries * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.4,
        f"{val}\n({pct:.0f}%)",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )

ax.set_xlabel("Budget k (simultaneous perturbations)", fontsize=12)
ax.set_ylabel("Unidentifiable queries resolved", fontsize=12)
ax.set_title(
    "Queries Resolved by Greedy-Optimal Panel at Each Budget\nE. coli Gene Regulatory Network",
    fontsize=13,
    fontweight="bold",
    pad=15,
)
ax.set_ylim(0, n_queries * 1.2)
ax.legend(fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "budget_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: budget_comparison.png")

## 6. Minimum Set Cover

What is the **fewest** number of TFs needed to resolve **all** resolvable queries?

In [ ]:
min_cover = greedy_min_set_cover(candidates, queries, matrix)

min_cover_rows = []
for rank, (tf, gain, cumulative) in enumerate(min_cover, start=1):
    min_cover_rows.append(
        {
            "Rank": rank,
            "Candidate TF": tf,
            "Marginal gain": gain,
            "Cumulative resolved": cumulative,
            "% of resolvable": f"{cumulative / n_resolvable * 100:.1f}%"
            if n_resolvable > 0
            else "N/A",
        }
    )

min_cover_df = pd.DataFrame(min_cover_rows)
min_cover_df.to_csv(OUTPUT_DIR / "nomination_min_cover.csv", index=False)

print(f"Minimum set cover: {len(min_cover)} TFs cover {n_resolvable} resolvable queries")
print()
display(min_cover_df)

## 7. Query-Level Heatmap

Which queries are resolved by which nominated TFs (for the recommended budget)?

In [ ]:
HEATMAP_K = 10  # adjust as needed

selected_tfs = [tf for tf, _, _ in full_greedy[:HEATMAP_K]]

# Build heatmap matrix: rows = selected TFs, cols = queries
heatmap_data = np.array([[int(matrix[tf][qi]) for qi in range(n_queries)] for tf in selected_tfs])

# Sort queries: first those resolved by the panel, then unresolved
panel_resolved = np.any(heatmap_data, axis=0)
col_order = np.argsort(~panel_resolved)  # resolved first
heatmap_sorted = heatmap_data[:, col_order]
query_labels_sorted = [queries[i] for i in col_order]

fig, ax = plt.subplots(figsize=(max(12, n_queries * 0.25), max(4, HEATMAP_K * 0.5)))

im = ax.imshow(heatmap_sorted, aspect="auto", cmap="Blues", vmin=0, vmax=1)

ax.set_yticks(range(HEATMAP_K))
ax.set_yticklabels([f"{i + 1}. {tf}" for i, tf in enumerate(selected_tfs)], fontsize=9)
ax.set_xticks(range(n_queries))
ax.set_xticklabels(query_labels_sorted, rotation=90, fontsize=7)
ax.set_xlabel("Unidentifiable query (do(TF1) -> outcome)", fontsize=11)
ax.set_ylabel("Nominated background TF (greedy rank)", fontsize=11)
ax.set_title(
    f"Coverage Heatmap: Top-{HEATMAP_K} Greedy Panel vs. All Queries\n"
    "Blue = query resolved by this background TF",
    fontsize=12,
    fontweight="bold",
    pad=12,
)

# Vertical line separating resolved / unresolved
n_panel_resolved = int(panel_resolved.sum())
ax.axvline(n_panel_resolved - 0.5, color=ORANGE, linewidth=2, linestyle="--")
ax.text(
    n_panel_resolved - 0.5,
    -1.5,
    f"<-- {n_panel_resolved} resolved | {n_queries - n_panel_resolved} unresolved -->",
    ha="center",
    fontsize=9,
    color=ORANGE,
)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"coverage_heatmap_k{HEATMAP_K}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: coverage_heatmap_k{HEATMAP_K}.png")

## 8. Decision Support Table

Summary table for the mapSPLiT screen design decision.

In [ ]:
decision_rows = []
for k in BUDGETS:
    df = nomination_tables[k]
    resolved = int(df["Cumulative resolved"].max()) if not df.empty else 0
    pct_unid = resolved / n_queries * 100
    pct_resol = resolved / n_resolvable * 100 if n_resolvable > 0 else 0
    top_tfs = ", ".join(df["Candidate TF"].tolist()[:3]) + ("..." if k > 3 else "")
    decision_rows.append(
        {
            "k (simultaneous perturbations)": k,
            "Queries resolved": resolved,
            "% of 55 unidentifiable": f"{pct_unid:.1f}%",
            "% of resolvable ceiling": f"{pct_resol:.1f}%",
            "Top nominated TFs": top_tfs,
        }
    )

decision_df = pd.DataFrame(decision_rows)
decision_df.to_csv(OUTPUT_DIR / "decision_support_table.csv", index=False)

print("Decision Support Table for mapSPLiT Screen Design")
print("=" * 80)
display(decision_df)
print()
print(f"Note: 'resolvable ceiling' = {n_resolvable}/{n_queries} queries that can be")
print("      resolved by at least one single background TF.")
print(f"      The remaining {n_queries - n_resolvable} queries require multi-background")
print("      interventions (Layer 2, future work) or are structurally unresolvable.")

## 9. Save All Outputs

Summary of all files written by this notebook.

In [ ]:
outputs = (
    [
        "coverage_distribution.png",
        # Section 2.5: cycle-breaking analysis
        "cycle_breaking_analysis.png",
        "cycle_breaking_analysis.csv",
        "cycle_vs_coverage_ranking.csv",
        # Section 3: marginal gain curve
        "marginal_gain_curve.png",
        "marginal_gain_curve.csv",
        # Section 5: budget comparison
        "budget_comparison.png",
        # Section 6: min set cover
        "nomination_min_cover.csv",
        # Section 8: decision support
        "decision_support_table.csv",
    ]
    + [f"nomination_k{k:02d}.csv" for k in BUDGETS]
    + [f"coverage_heatmap_k{HEATMAP_K}.png"]
)

print("Files written to:", OUTPUT_DIR)
for fname in outputs:
    fpath = OUTPUT_DIR / fname
    status = "OK" if fpath.exists() else "MISSING"
    print(f"  [{status}] {fname}")